# Order constraints

Sortedness states the order of one column against the row sequence,
and functional dependencies state when one column's value fixes
another's. Neither states the row-level order between two columns:
that an end does not precede its start, that a part does not
outweigh the whole. The `order_constraints` pass states exactly
that. It screens pairs of columns that can be compared exactly —
numeric with numeric, dates with dates, datetimes with datetimes
of one unit — verifies a direction `a <= b` on every row where both
columns are present, and reports the direction when it holds
exactly or fails on at most a small counted share of the rows.

This page is about reading those statements: what one of them
establishes, what its evidence carries, which pairs the pass
screens and which it never sees, and how to tell its silences
apart. The worked example is the satellites table from the corpus
— 1,164 rows describing catalogued satellites — profiled with the
detector list its corpus entry names.

In [1]:
import json

import polars as pl

import onceover

ROOT = "../../../benchmarks/corpus"
registry = json.load(open(f"{ROOT}/registry.json"))
entry = next(e for e in registry["datasets"] if e["id"] == "lace-satellites")

df = pl.read_csv(
    f"{ROOT}/data/crosscat/lace-satellites/data.csv",
    infer_schema_length=None,
    try_parse_dates=bool(entry.get("parse_dates")),
)
report = onceover.profile(df, budget=entry["budget"], detectors=entry["detectors"])

orders = [f for f in report.findings if f["type"] == "order_constraint"]
print(f"{df.height} rows, {df.width} columns; profiled at the {entry['budget']} budget "
      f"with the corpus entry's {len(entry['detectors'])} detectors")
print(f"{len(orders)} order statements:")
for f in orders:
    print("-", f["claim"])
print()
print("coverage:", report.data["coverage"]["order_constraints"]["detail"])

1164 rows, 21 columns; profiled at the standard budget with the corpus entry's 19 detectors
3 order statements:
- Perigee_km <= Apogee_km on all 1163 evaluated rows (0 violations; the two are equal on 17 of them)
- Period_minutes <= Apogee_km on all 1162 evaluated rows (0 violations; the two are equal on 0 of them; the tightest margin is 226.39)
- Dry_Mass_kg <= Launch_Mass_kg on all 480 evaluated rows (0 violations; the two are equal on 13 of them)

coverage: 23 of 23 eligible column pairs screened; 6 of 6 that stayed near-universal verified on all rows on the shared max_pairs budget; 3 reportable order directions, 3 findings


The three statements are claims about rows, not about the
columns' ranges. `Perigee_km <= Apogee_km` says that on every one
of the 1,163 rows where both distances are recorded, the lower
point of the orbit does not exceed the higher point. The two are
equal on 17 of those rows — near-circular orbits, where the
relation holds as an equality. One of the 1,164 rows is not
evaluated at all: it has no pair to compare, and the evidence
counts it apart as a missing row rather than letting it vote
either way.

`Dry_Mass_kg <= Launch_Mass_kg` is evaluated on fewer than half
the rows: 480 of 1,164 record both masses. The statement's scope
is those 480 rows, and the evidence says so — a statement about a
partly recorded pair covers the rows where the pair is complete,
and no others.

`Period_minutes <= Apogee_km` pairs minutes with kilometres. The
pass compares values, not meanings: the two columns share the
numeric kind and their ranges overlap, so the direction is
tested, and it holds with room to spare — the tightest margin,
the smallest gap between the two values on any row, is 226.39.
Whether a period bounding an apogee says anything about
satellites is the reader's judgement. What the pass establishes
is that the relation lives in how the values pair up in rows,
not in the two ranges: randomly re-pairing the two columns would
break it 240.6 times on average. That shuffled count stands in
each statement's evidence as the control the direction had to
beat — 455.5 for the perigee pair, 165.4 for the masses.

In [2]:
import altair as alt

pts = df.select("Perigee_km", "Apogee_km").drop_nulls()
line = pl.DataFrame({"x": [150.0, 500_000.0], "y": [150.0, 500_000.0]})
points = (
    alt.Chart(pts).mark_circle(size=18, opacity=0.55)
    .encode(
        x=alt.X("Perigee_km:Q", scale=alt.Scale(type="log"), title="Perigee (km, log scale)"),
        y=alt.Y("Apogee_km:Q", scale=alt.Scale(type="log"), title="Apogee (km, log scale)"),
    )
)
rule = alt.Chart(line).mark_line(color="#d7301f").encode(x="x:Q", y="y:Q")
points + rule

alt.LayerChart(...)

What to look for: every point sits on or above the diagonal.
Points on the line are the 17 ties. A point below the line would
be a violation, and its distance below it would be the size of
the violation in the pair's own units.

In [3]:
gaps = (
    df.select((pl.col("Apogee_km") - pl.col("Period_minutes")).alias("gap"))
    .drop_nulls()
    .filter(pl.col("gap") > 0)
)
(alt.Chart(gaps).mark_bar()
 .encode(x=alt.X("gap:Q", bin=alt.Bin(maxbins=40),
                 title="Apogee_km - Period_minutes"),
         y=alt.Y("count()", title="Rows"))
 .properties(height=220))

alt.Chart(...)

What to look for: the distribution starts at 226.39, not at
zero. That left edge is the tightest margin — the smallest amount
by which the statement clears its closest row. A row whose gap
crossed zero would be a violation, so the margin is the distance
between the statement and its first exception.

## What a statement establishes — and what it does not

A statement is a property of the dataset as given. It says the
relation held on every evaluated row of this table, and nothing
about the domain the table describes. That a perigee never
exceeds an apogee is a fact of orbital geometry; the pass does
not know it, and the statement would read identically for a
relation that holds only because of how this catalogue was
assembled. If a source schema declares the order, the statement
confirms it on the measured rows; if it does not, the statement
marks a regularity whose first future violation is a data-error
signal. That is the finding's own interpretation, and it is the
strongest reading the evidence supports.

Two boundaries of the claim are worth keeping in view. The claim
covers the rows where both columns are present; the missing rows
are counted in the evidence, not covered by the claim. And the
claim is exact as a count, not approximate as a tendency: the
direction is verified on all rows before it is reported, so a
statement is never a summary of a sample. What a single violation
does to it is the next section.

In [4]:
row = (df.with_row_index().drop_nulls("Apogee_km")
       .sort("Apogee_km", descending=True)["index"][0])
planted = df.with_columns(
    pl.when(pl.int_range(pl.len()) == row)
    .then(pl.col("Apogee_km") + 100.0)
    .otherwise(pl.col("Perigee_km"))
    .alias("Perigee_km")
)
planted_report = onceover.profile(planted, detectors=["order_constraints"])
for f in planted_report.findings:
    if f["type"] == "order_constraint":
        print("-", f["claim"])

- Perigee_km <= Apogee_km on 1162 of 1163 evaluated rows (1 violations, 0.09%): the worst violation is at row 1101, where Perigee_km = 470410 against Apogee_km = 470310
- Period_minutes <= Perigee_km on 1151 of 1162 evaluated rows (11 violations, 0.95%): the worst violation is at row 483, where Period_minutes = 4032.86 against Perigee_km = 2284
- Period_minutes <= Apogee_km on all 1162 evaluated rows (0 violations; the two are equal on 0 of them; the tightest margin is 226.39)
- Dry_Mass_kg <= Launch_Mass_kg on all 480 evaluated rows (0 violations; the two are equal on 13 of them)


One changed cell — the perigee of the satellite with the
highest apogee in the table (row 1101: perigee 186 km against an
apogee of 470,310 km), set 100 above its apogee — demotes the
first statement from exact to approximate. It does not erase it.
A direction is reported while its violations are zero or at most
1% of the evaluated rows, and the approximate statement carries
its exceptions as its content: the count, the share, the first
ten violating rows, and the worst violation with its row and
both values. An end recorded before its start is the common
shape of the error this form exists to catch, and the exceptions
are the rows a reader would inspect.

The same cell also moves a pair the change did not aim at.
`Period_minutes <= Perigee_km` failed on 12 rows of the original
table — 1.03%, just over the bar, so it earned no statement —
and raising that one perigee satisfies one of its violations,
bringing it to 11 (0.95%), inside the bar. Pairs that share a
column are not independent exhibits: one cell can move several
of them across the line in either direction.

## Which pairs are screened

A pair becomes a candidate only when its columns can be compared
exactly and the comparison can say something about rows:

- The columns share a kind family: numeric with numeric, dates
  with dates, datetimes with datetimes of one unit. Integers
  compare as integers and floats as floats; a mixed pair compares
  as floats only when the integer side is exactly representable
  in floating point.
- Each column carries at least 100 non-null values and at least
  5 distinct values. Below the distinct floor the pair is a
  levels question for the dependency passes, and a constant
  column would make any order a statement about one column's
  range.
- The marginal ranges are compatible: they overlap by at least a
  quarter of the narrower range, and their widths are within
  100x. When one column's maximum does not exceed the other's
  minimum, `a <= b` follows from the two ranges alone, whatever
  the rows do. When the widths differ by orders of magnitude, the
  columns measure different quantities, and a near-universal
  order between them is an artifact of scale.
- Columns read as row counters, identifiers, or calendar parts
  are never paired: their order against another column says
  nothing about the data.

Candidates are screened nearest columns first. On tables past
4,096 rows the screen runs on a deterministic row sample, with a
survival bar of twice the reporting bar; survivors are verified
on all rows. The verification budget is the shared pair budget —
496 pairs at the standard budget — and the pass emits at most 25
findings. Coverage states how many pairs were screened, how many
survivors were verified, and how many directions were
reportable, and it reads `capped` when a budget binds. At the
quick budget the pair budget is zero and the pass records itself
as skipped. The cell below recomputes the satellites screen from
the frame itself, with the same guards and bars.

In [5]:
numeric = [name for name, dtype in df.schema.items() if dtype == pl.Float64]
values = {name: df[name].to_list() for name in numeric}


def shuffled_violations(lower, upper):
    pairs = [(x, y) for x, y in zip(lower, upper) if x is not None and y is not None]
    xs = sorted(x for x, _ in pairs)
    ys = sorted(y for _, y in pairs)
    below = crossed = 0
    for x in xs:
        while below < len(ys) and ys[below] < x:
            below += 1
        crossed += below
    return crossed / max(len(xs), 1)


eligible = 0
rows = []
for i, a in enumerate(numeric):
    for b in numeric[i + 1:]:
        A = [v for v in values[a] if v is not None]
        B = [v for v in values[b] if v is not None]
        if len(A) < 100 or len(B) < 100 or len(set(A)) < 5 or len(set(B)) < 5:
            continue
        wa, wb = max(A) - min(A), max(B) - min(B)
        narrow, wide = min(wa, wb), max(wa, wb)
        overlap = min(max(A), max(B)) - max(min(A), min(B))
        if wide > 100 * narrow or overlap < 0.25 * narrow:
            continue
        eligible += 1
        pairs = [(x, y) for x, y in zip(values[a], values[b])
                 if x is not None and y is not None]
        vab = sum(1 for x, y in pairs if x > y)
        vba = sum(1 for x, y in pairs if y > x)
        if min(vab, vba) > 0.02 * len(pairs):
            continue
        lower, upper, viol = (a, b, vab) if vab <= vba else (b, a, vba)
        rows.append({"pair": f"{lower} <= {upper}", "rows": len(pairs),
                     "violations": viol, "share": viol / len(pairs),
                     "shuffled": round(shuffled_violations(values[lower], values[upper]), 1)})
survivors = pl.DataFrame(rows).sort("share")
print(f"{len(numeric)} numeric columns, {len(numeric) * (len(numeric) - 1) // 2} pairs "
      f"before the guards; {eligible} eligible; {survivors.height} survive the screen:")
print(survivors)

11 numeric columns, 55 pairs before the guards; 23 eligible; 6 survive the screen:
┌─────────────────────────────────┬──────┬────────────┬──────────┬──────────┐
│ pair                            ┆ rows ┆ violations ┆ share    ┆ shuffled │
╞═════════════════════════════════╪══════╪════════════╪══════════╪══════════╡
│ Perigee_km <= Apogee_km         ┆ 1163 ┆ 0          ┆ 0.0      ┆ 455.5    │
│ Period_minutes <= Apogee_km     ┆ 1162 ┆ 0          ┆ 0.0      ┆ 240.6    │
│ Dry_Mass_kg <= Launch_Mass_kg   ┆ 480  ┆ 0          ┆ 0.0      ┆ 165.4    │
│ Eccentricity <= Expected_Lifet… ┆ 877  ┆ 4          ┆ 0.004561 ┆ 1.8      │
│ Period_minutes <= Perigee_km    ┆ 1162 ┆ 12         ┆ 0.010327 ┆ 257.6    │
│ longitude_radians_of_geo <= Ex… ┆ 380  ┆ 7          ┆ 0.018421 ┆ 5.2      │
└─────────────────────────────────┴──────┴────────────┴──────────┴──────────┘


Three of the six survivors earn no statement, each stopped
by a different bar. `Period_minutes <= Perigee_km` is verified
with 12 violations in 1,162 rows — 1.03%, just over the 1%
reporting bar. `Eccentricity <= Expected_Lifetime` sits inside
the bar at 4 violations (0.46%), but random pairing of the same
two columns would produce only 1.8 violations: a column bounded
by 1 set against lifetimes running to 30 years crosses rarely
however the rows are paired, so the near-universal order is
already in the two marginals and proves nothing about rows.
`longitude_radians_of_geo <= Expected_Lifetime` fails on both
counts, at 1.84% violations and a shuffled count of 5.2. A
reported direction must clear all three bars: near-universal on
the screen, within 1% on all rows, and at least four times
cleaner than random pairing, which must itself promise at least
20 violations.

## How statements rank

When more directions pass than a reader will look at first,
order findings rank in three classes. A direction with
violations ranks first — the exceptions to a near-universal
order are the data-error signal. Exact orders between columns
with no functional relation rank next. An exact order between
columns that an exact functional dependency ties in the same run
ranks last and carries `functionally_related` in its evidence:
when one column is a function of the other, the order may follow
from the derivation rather than from a constraint on the source.
The test suite's ranking plant — a discrete base and its derived
double, an unrelated exact pair, and a pair with 25 planted
violations in 5,000 rows — produces all three classes in one
report:

In [6]:
import random

rng = random.Random(8)
n = 5000
base = [float(rng.randrange(1, 200)) for _ in range(n)]
lo = [rng.uniform(0.0, 50.0) for _ in range(n)]
hi = [x + rng.uniform(0.0, 50.0) for x in lo]
lo_v, hi_v = list(lo), list(hi)
for i in rng.sample(range(n), 25):
    lo_v[i] = hi_v[i] + rng.uniform(0.1, 5.0)
plant = pl.DataFrame({
    "base": base,
    "derived": [2.0 * x + 3.0 for x in base],
    "lo": lo, "hi": hi, "lo_v": lo_v, "hi_v": hi_v,
})
plant_report = onceover.profile(plant)
for f in plant_report.findings:
    if f["type"] == "order_constraint":
        m = f["evidence"][0]["metrics"]
        print(f'{m["lower"]} <= {m["upper"]}: interestingness {f["interestingness"]}, '
              f'violations {m["violations"]}, functionally_related {m["functionally_related"]}')

lo_v <= hi: interestingness 0.8936, violations 25, functionally_related False
lo_v <= hi_v: interestingness 0.8936, violations 25, functionally_related False
lo <= hi: interestingness 0.8045, violations 0, functionally_related False
lo <= hi_v: interestingness 0.8045, violations 0, functionally_related False
lo <= lo_v: interestingness 0.8045, violations 0, functionally_related False
base <= derived: interestingness 0.7195, violations 0, functionally_related True


The two violated directions lead, the unrelated exact orders
follow, and `base <= derived` closes the list at the lowest
interestingness, flagged as functionally related. The ranking is
part of the reading: the report shows the orders most likely to
be errors first, and the orders most likely to be restatements
of a derivation last.

## When neither direction holds

The certification bar is not the only statement a pair can
earn. When the less-violated direction fails on at least 20
rows and at least 5% of the evaluated rows, while the
reverse direction's violations are at least twice as many,
the pair is reported as a systematic violation of the
less-violated direction: the claim states the violating row
count and share, the reverse count and share, and the worst
violation, and the violating rows are listed in the
evidence. The statement also requires the two columns to
track each other on a comparable scale — the median row
distance |a − b| / max(|a|, |b|) over sampled complete rows
is at most 0.2, and the columns' medians agree within a
factor of 2.5 when both are non-zero. Two unrelated columns
crossing in both directions at similar rates fail the
asymmetry term. An age in years against a duration in
seconds crosses asymmetrically, but from far off the
diagonal, and fails the scale term. What passes is the
shape of a bound that held by construction — an accepted
count against the generated count it is bounded by — and
is broken in bulk: a definition that changed partway
through the data, or two populations merged under one pair
of columns.

In [7]:
# accepted is bounded by generated by construction — and
# exceeds it on 480 of 2,000 rows.
generated = [50 + (k * 37) % 200 for k in range(2000)]
accepted = [
    g + 5 if k < 480 else g - 3 if k < 1780 else g
    for k, g in enumerate(generated)
]
counts = pl.DataFrame({"generated": generated, "accepted": accepted})
counts_report = onceover.profile(counts)
hit = next(f for f in counts_report.findings
           if f["type"] == "order_violation")
print(hit["claim"])
print("violations:", hit["evidence"][0]["metrics"]["violations"])

accepted <= generated is violated on 480 of 2000 evaluated rows (24.0%), while the reverse order is violated on 1300 rows (65.0%): the worst violation is at row 0, where accepted = 55 against generated = 50
violations: 480


## The three silences

No statement about a pair can mean three different things, and
only the coverage entry tells them apart.

In [8]:
rng = random.Random(4)
lo = [rng.uniform(0, 50) for _ in range(60)]
small = pl.DataFrame({"lo": lo, "hi": [x + rng.uniform(0, 50) for x in lo]})
small_report = onceover.profile(small, detectors=["order_constraints"])
print("findings:", len(small_report.findings))
print("coverage:", small_report.data["coverage"]["order_constraints"]["detail"])

findings: 0
coverage: 0 of 0 eligible column pairs screened; 0 of 0 that stayed near-universal verified on all rows on the shared max_pairs budget; 0 reportable order directions, 0 findings


Never eligible. The order on this 60-row frame is exact, and
the pass says nothing: each column sits below the 100 non-null
floor, so the pair is never a candidate and coverage counts 0 of
0 eligible pairs. In the findings list, the silence of a pair
that was never examined reads exactly like the silence of one
that was examined and failed; the difference exists only in
coverage. Pairs of different kinds, pairs whose ranges do not
overlap, and pairs involving a text column are silent the same
way — the last of those is the filings case below, at the scale
of a whole table.

In [9]:
base = [rng.uniform(0, 7) for _ in range(2000)]
identical = pl.DataFrame({"base": base, "base_copy": list(base)})
identical_report = onceover.profile(identical)
print("order coverage:",
      identical_report.data["coverage"]["order_constraints"]["detail"])
for f in identical_report.findings:
    if f["type"] == "functional":
        print("-", f["claim"])

order coverage: 1 of 1 eligible column pairs screened; 1 of 1 that stayed near-universal verified on all rows on the shared max_pairs budget; 0 reportable order directions, 0 findings
- base_copy = 1.000000·base within tolerance on all 2000 non-missing row pairs


Stated elsewhere. Two identical columns satisfy `a <= b` in
both directions with zero violations, and the pass states no
order: when both directions pass with equal violations, the pair
is an equality, not an order. The report does not go quiet on
the pair — the functional-dependency pass states the equality as
an exact relation, which is the claim the data supports. The
same rule retires pairs that cross as often as they agree: equal
nonzero violation counts in the two directions are a
near-equality, not a near-universal order.

In [10]:
rng = random.Random(11)
n = 5000
tail = [rng.gauss(0, 1) / max(abs(rng.gauss(0, 1)), 1e-3) for _ in range(n)]
count = [rng.randint(0, 1000) for _ in range(n)]
heavy = pl.DataFrame({"tail": tail, "count": count})
heavy_report = onceover.profile(heavy, detectors=["order_constraints"])
observed = sum(1 for x, y in zip(tail, count) if x > y)
print("order findings:", len(heavy_report.findings))
print("coverage:", heavy_report.data["coverage"]["order_constraints"]["detail"])
print(f"tail <= count is violated on {observed} of {n} rows; random pairing of the "
      f"same columns would violate it {shuffled_violations(tail, count):.1f} times")

order findings: 0
coverage: 1 of 1 eligible column pairs screened; 1 of 1 that stayed near-universal verified on all rows on the shared max_pairs budget; 0 reportable order directions, 0 findings; pairs screened on a deterministic row sample and verified on all rows
tail <= count is violated on 15 of 5000 rows; random pairing of the same columns would violate it 14.5 times


Screened, verified, and refused by the chance bar. The
heavy-tailed column crosses the count column on only 15 of 5,000
rows — inside the 1% bar — but random pairing would cross it
only 14.5 times, so the near-universal order is a fact about the
two marginal distributions, not about how values pair in rows.
This is the control the satellites statements passed with
shuffled counts of 165 to 455. Before it existed, independent
columns of this shape filled the findings cap on null frames;
the detector's benchmark record carries that history.

In [11]:
edgar = next(e for e in registry["datasets"] if e["id"] == "sec-edgar-company-facts")
filings = pl.read_csv(
    f"{ROOT}/data/business/sec-edgar-company-facts/data.csv",
    infer_schema_length=None,
    try_parse_dates=bool(edgar.get("parse_dates")),
)
filings_report = onceover.profile(filings, budget=edgar["budget"])
print("period_end:", filings.schema["period_end"], "| filed:", filings.schema["filed"])
print("order coverage:",
      filings_report.data["coverage"]["order_constraints"]["detail"])
parsed = filings.select(
    pl.col("period_end").str.to_date().alias("period_end"),
    pl.col("filed").str.to_date().alias("filed"),
).drop_nulls()
violations = parsed.filter(pl.col("period_end") > pl.col("filed")).height
print(f"read as dates by hand: period_end <= filed is violated on {violations} "
      f"of {parsed.height} rows")

period_end: String | filed: String
order coverage: 0 of 0 eligible column pairs screened; 0 of 0 that stayed near-universal verified on all rows on the shared max_pairs budget; 0 reportable order directions, 0 findings; pairs screened on a deterministic row sample and verified on all rows
read as dates by hand: period_end <= filed is violated on 0 of 11543 rows


The corpus's company-filings table carries the relation a
reader would most expect this pass to state: a filing's period
cannot end after the filing was made. Read as dates,
`period_end <= filed` holds on all 11,543 rows, with no
violations and no ties — the last line above is that check done
by hand. The pass states nothing, and its coverage counts 0 of 0
eligible pairs. The corpus read takes both columns as text, and
text columns are never paired; the table's three numeric columns
— a company identifier, a fiscal year, and the reported values —
share no compatible range, so no pair among them is eligible
either.

The relation is true, the data is clean, and the check is blind
to it: eligibility is a property of the columns as read, not of
the values they encode. A reader who parses the dates before
profiling brings the pair into the pass's universe. A reader who
does not gets a silence that only the coverage entry
distinguishes from a table with nothing to state.

## Reading an order statement

- The direction and its scope. The claim names the lower and
  upper columns and the rows where both are present; the
  evidence adds the table's row count and the missing rows the
  claim does not cover.
- Exact or approximate. Zero violations, or a counted share of
  at most 1%, with the violating rows listed and the worst
  violation stated with its row and both values.
- The ties. Rows where the two columns are equal satisfy the
  order and are counted separately from the rows that clear it
  strictly.
- The margin. An exact statement's tightest margin is the
  smallest gap on any row — the distance between the statement
  and its first violation.
- The control. `violations_if_shuffled` counts the violations
  random pairing of the same columns would produce; a reported
  direction beat it by at least four times, from a floor of 20.
- The flag. `functionally_related` marks a pair that an exact
  functional dependency ties in the same run; its statement
  ranks below unrelated pairs, and every statement with
  violations ranks above both.
- The coverage. How many pairs were screened, how many were
  verified, and whether a budget capped either — the only place
  the three silences can be told apart.